[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Smiledxd/python_data/blob/main/sesiones/S06_numpy_2d_axis.ipynb)

# Sesión 06 · Arrays 2D y `axis`

**Módulo 2: NumPy** · ⏱️ Duración estimada: 60 minutos

## 🎯 Objetivos
Al terminar esta sesión podrás:
1. Crear arrays 2D, cambiarles la forma con `reshape` (también con `-1`) y transponerlos.
2. Operar una matriz con una fila y con una columna `(n, 1)` usando broadcasting.
3. Resumir por filas o por columnas con `axis=0` y `axis=1`, y conservar la forma con `keepdims`.
4. Estandarizar cada columna con el z-score.

## 📋 Qué debes saber antes
Sesiones 4 y 5: crear arrays, broadcasting con escalares, agregaciones y máscaras.

## 🧭 Cómo trabajar este notebook
- Ejecuta las celdas **en orden**, de arriba abajo, con **Shift + Enter**.
- En cada ✍️ **Tu turno** escribe tu código debajo de `# Tu código aquí`.
- Después ejecuta la celda ✅ **Verificar**. Si aparece ❌, lee el motivo, corrige y vuelve a verificar.
- Si te atascas, abre la 💡 **Pista**. Hay dos, de menos a más ayuda.
- Truco para toda la sesión: cuando dudes, imprime `.shape`. La mayoría de los errores con 2D son formas que no encajan.

## ⚙️ Setup
Ejecuta la celda siguiente **al empezar** (y otra vez si reinicias el entorno). Genera los datos de práctica y carga las funciones que revisan tus respuestas.

⚠️ **Ejecútala y no la edites.**

In [ ]:
#@title ⚙️ Setup: ejecuta esta celda y no la edites { display-mode: "form" }
# Prepara los datos de la sesión y las funciones que revisan tus respuestas.
import copy
import hashlib
import math
import statistics

import numpy as np

rng = np.random.default_rng(42)

# ---------- Datos de práctica: ventas de tiendas ----------
nombres_tiendas = np.array(["Miraflores", "Surco", "Lince", "Barranco", "San Isidro"])
meses = np.array(["ene", "feb", "mar", "abr", "may", "jun"])
ventas_flat = rng.integers(20, 90, size=30) * 1000      # 6 meses de la 1.ª tienda, luego 6 de la 2.ª, etc.
metas_mes = np.array([50, 50, 55, 55, 60, 70]) * 1000
descuento_tienda = np.array([0.05, 0.10, 0.0, 0.08, 0.12])

productos = np.array(["polo", "jean", "casaca", "gorra"])
precios_prod = np.array([39.9, 119.9, 189.9, 25.0])
unidades_tp = rng.integers(0, 60, size=(5, 4))          # filas: tiendas; columnas: productos

# ---------- Datos de práctica: clientes de un banco ----------
variables = np.array(["ingreso", "gasto", "operaciones", "antiguedad"])
clientes_feat = np.column_stack([
    np.round(rng.normal(3500, 1200, 8), 2),
    np.round(rng.normal(2200, 800, 8), 2),
    rng.integers(5, 80, 8).astype(float),
    rng.integers(1, 15, 8).astype(float),
])
saldos = np.round(np.clip(rng.normal(5000, 2000, size=(6, 12)), 200, None), 2)   # 6 clientes × 12 meses

_NOMBRES = ["nombres_tiendas", "meses", "ventas_flat", "metas_mes", "descuento_tienda", "productos",
            "precios_prod", "unidades_tp", "variables", "clientes_feat", "saldos"]
_D = copy.deepcopy({k: globals()[k] for k in _NOMBRES})
# Versiones en listas de Python: los verificadores recalculan con bucles, sin NumPy.
_L = {k: (v.tolist() if isinstance(v, np.ndarray) else v) for k, v in _D.items()}

# ---------- Herramientas de verificación ----------
_FALTA = object()


def _h(valor):
    if isinstance(valor, str):
        valor = valor.strip().lower()
    return hashlib.sha256(f"{type(valor).__name__}|{valor!r}".encode("utf-8")).hexdigest()


def _cerca(a, b, tol=1e-9):
    return math.isclose(a, b, rel_tol=1e-9, abs_tol=tol)


def _dos_decimales(x):
    return abs(x * 100 - round(x * 100)) < 1e-6


def _corto(valor, n=60):
    if type(valor).__module__ == "numpy" and getattr(valor, "shape", None) == ():
        valor = valor.item()
    texto = repr(valor)
    return texto if len(texto) <= n else texto[:n] + "…"


def _igual(a, b, tol=1e-6):
    """Compara exigiendo el mismo tipo en None/bool y tolerancia en decimales."""
    if b is None or isinstance(b, bool):
        return type(a) is type(b) and a == b
    if isinstance(b, (int, float)) and not isinstance(b, bool):
        return (isinstance(a, (int, float)) and not isinstance(a, bool)
                and math.isclose(a, b, rel_tol=1e-9, abs_tol=tol))
    if isinstance(b, (list, tuple)):
        return (type(a) is type(b) and len(a) == len(b)
                and all(_igual(x, y, tol) for x, y in zip(a, b)))
    if isinstance(b, dict):
        return (isinstance(a, dict) and set(a) == set(b)
                and all(_igual(a[k], b[k], tol) for k in b))
    return type(a) is type(b) and a == b


class _Revision:
    def __init__(self, titulo):
        self.titulo = titulo
        self.errores = 0
        print(f"── {titulo} ──")

    def ok(self, msg):
        print(f"✅ {msg}")

    def mal(self, msg):
        self.errores += 1
        print(f"❌ {msg}")

    def var(self, nombre, tipo=None):
        valor = globals().get(nombre, _FALTA)
        if valor is _FALTA:
            self.mal(f"No encuentro `{nombre}`. ¿Ejecutaste tu celda? ¿Escribiste bien el nombre?")
            return _FALTA
        if tipo is not None and not (type(valor) is tipo or (isinstance(tipo, tuple) and type(valor) in tipo)):
            esperado = tipo.__name__ if not isinstance(tipo, tuple) else " o ".join(t.__name__ for t in tipo)
            self.mal(f"`{nombre}` es de tipo {type(valor).__name__} y se esperaba {esperado}.")
            return _FALTA
        return valor

    def funcion(self, nombre):
        f = self.var(nombre)
        if f is _FALTA:
            return _FALTA
        if not callable(f):
            self.mal(f"`{nombre}` existe pero no es una función. ¿La definiste con `def`?")
            return _FALTA
        return f

    def caso(self, texto, f, args=(), kwargs=None, esperado=None, igual=None, motivo="no es lo esperado", tol=0.0051):
        """Llama a f con copias de los argumentos y compara sin mostrar el valor esperado."""
        import copy
        try:
            obtenido = f(*copy.deepcopy(args), **copy.deepcopy(kwargs or {}))
        except Exception as e:
            self.mal(f"`{texto}` lanzó {type(e).__name__}: {e}")
            return False
        bien = igual(obtenido, esperado) if igual else _igual(obtenido, esperado, tol)
        if bien:
            self.ok(f"`{texto}` funciona.")
        else:
            self.mal(f"`{texto}` devolvió {_corto(obtenido)}; {motivo}.")
        return bien

    def valor(self, nombre, esperado, tipo=None, pista="revisa el cálculo", igual=None):
        v = self.var(nombre, tipo)
        if v is _FALTA:
            return
        bien = igual(v, esperado) if igual else _igual(v, esperado)
        if bien:
            self.ok(f"`{nombre}` es correcto.")
        else:
            self.mal(f"`{nombre}` vale {_corto(v)}; {pista}.")

    def texto_limpio(self, nombre, valor):
        if valor != valor.strip():
            self.mal(f"`{nombre}` tiene espacios o saltos de línea al inicio o al final: {valor!r}")
            return False
        return True

    def predicciones(self, esperados):
        for nombre, hash_ok in esperados.items():
            v = self.var(nombre)
            if v is _FALTA:
                continue
            if _h(v) == hash_ok:
                self.ok(f"`{nombre}` es correcto.")
            else:
                self.mal(f"`{nombre}` no es correcto. Razónalo otra vez y luego compruébalo ejecutando la expresión en una celda nueva.")

    def fin(self):
        if self.errores == 0:
            print(f"🎉 ¡{self.titulo} superado!")
        else:
            cuantos = "el punto marcado" if self.errores == 1 else f"los {self.errores} puntos marcados"
            print(f"🔁 Corrige {cuantos} con ❌ y vuelve a verificar.")


def _primera_diferencia(r, nombre, tuyo, esperado):
    for i, (a, b) in enumerate(zip(tuyo, esperado)):
        if a != b:
            r.mal(f"`{nombre}` no tiene el formato pedido. La diferencia empieza en el carácter {i}: "
                  f"desde ahí tu texto dice {tuyo[i:i + 15]!r}.")
            return
    n = abs(len(esperado) - len(tuyo))
    cuantos = "1 carácter" if n == 1 else f"{n} caracteres"
    if len(tuyo) < len(esperado):
        r.mal(f"`{nombre}` está incompleto: le {'falta' if n == 1 else 'faltan'} {cuantos} al final.")
    else:
        r.mal(f"`{nombre}` tiene {cuantos} de más al final: {tuyo[len(esperado):]!r}.")


def _es_numero(x):
    return isinstance(x, (int, float, np.integer, np.floating)) and not isinstance(x, (bool, np.bool_))


def _esc(r, nombre, esperado, pista, tol=1e-6):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if isinstance(v, np.ndarray) and v.shape == ():
        v = v.item()
    if not _es_numero(v):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un número.")
    elif abs(float(v) - esperado) <= tol:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` vale {_corto(v.item() if hasattr(v, 'item') else v)}; {pista}.")


def _arr(r, nombre, esperado, pista, tol=1e-6, tipos=None):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if not isinstance(v, np.ndarray):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un array de NumPy (`np.ndarray`).")
        return
    esperado = np.array(esperado)
    if v.shape != esperado.shape:
        r.mal(f"`{nombre}` tiene forma {v.shape} y se esperaba {esperado.shape}.")
        return
    if tipos and v.dtype.kind not in tipos:
        nombres = {"b": "bool", "i": "entero", "u": "entero", "f": "decimal (float)", "U": "texto"}
        r.mal(f"`{nombre}` tiene dtype {v.dtype} y se esperaba un tipo {' o '.join(sorted({nombres[t] for t in tipos}))}.")
        return
    if v.dtype.kind in "USO" or esperado.dtype.kind in "USO":
        bien = v.tolist() == esperado.tolist()
    elif v.dtype.kind == "b" or esperado.dtype.kind == "b":
        bien = np.array_equal(v, esperado)
    else:
        bien = np.allclose(v.astype(float), esperado.astype(float), rtol=0, atol=tol, equal_nan=True)
    if bien:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` tiene la forma correcta pero sus valores no coinciden; {pista}.")




def _sin_cambios(r, *nombres):
    for n in nombres:
        actual = globals().get(n)
        original = _D[n]
        if isinstance(original, np.ndarray):
            igual = isinstance(actual, np.ndarray) and actual.shape == original.shape and np.array_equal(actual, original, equal_nan=original.dtype.kind == "f")
        else:
            igual = actual == original
        if not igual:
            r.mal(f"`{n}` cambió. No modifiques los datos originales; vuelve a ejecutar el setup.")


def _tabla():
    v = _L["ventas_flat"]
    return [v[i * 6:(i + 1) * 6] for i in range(5)]


def _transpuesta(m):
    return [[fila[j] for fila in m] for j in range(len(m[0]))]


def _z_columnas(m):
    cols = _transpuesta(m)
    salida = []
    for col in cols:
        media, desv = statistics.fmean(col), statistics.pstdev(col)
        salida.append([0.0 if desv == 0 else (x - media) / desv for x in col])
    return _transpuesta(salida)


def check_ejercicio_1():
    r = _Revision("Ejercicio 1 · Parte A")
    t = _tabla()
    _arr(r, "tabla", t, "cada fila debería ser una tienda con sus 6 meses, en orden")
    _arr(r, "tabla_auto", t, "debería ser igual a `tabla`, pero usando `-1` en `reshape`")
    _arr(r, "por_mes", _transpuesta(t), "cada fila debería ser un mes con sus 5 tiendas")
    _arr(r, "columna", [[x] for x in _L["ventas_flat"]], "deberían ser las 30 ventas en una sola columna")
    _arr(r, "ceros_2d", [[0.0] * 4 for _ in range(3)], "deberían ser 3 filas y 4 columnas de ceros")
    _sin_cambios(r, "ventas_flat")
    r.fin()
    r = _Revision("Ejercicio 1 · Parte B")
    r.predicciones({
        "pred_reshape_4": "bd1dddfaf233665e87fb493c0364dc67523ff4525fc194616e411b16fa707f7a",
        "pred_forma_t": "37892b98c8a0ac5a3073950de71ccd7b3839a21f9741f817440dae1921fc6532",
        "pred_size_3d": "6abf698882849a1b7097609fbbf35547d9d5be79cf454b62f8f63fb0c9cfc898",
        "pred_ndim_col": "b4d4f68c2268549cada66d24ae3a1902d4152a98ad614bbf891edf235ef99218",
    })
    r.fin()


def check_ejercicio_2():
    r = _Revision("Ejercicio 2 · Parte A")
    t, metas = _tabla(), _L["metas_mes"]
    _arr(r, "ingresos_tp", [[u * p for u, p in zip(fila, _L["precios_prod"])] for fila in _L["unidades_tp"]],
         "cada columna de unidades debería multiplicarse por el precio de su producto")
    _arr(r, "vs_meta", [[x - m for x, m in zip(fila, metas)] for fila in t],
         "a cada venta réstale la meta de su mes (columna)")
    _arr(r, "alcanzo_meta", [[x >= m for x, m in zip(fila, metas)] for fila in t],
         "`True` donde la venta alcanza la meta de su mes", tipos="b")
    _arr(r, "con_descuento", [[x * (1 - d) for x in fila] for fila, d in zip(t, _L["descuento_tienda"])],
         "cada fila (tienda) debería usar su propio descuento; revisa la forma de columna `(5, 1)`")
    r.fin()
    r = _Revision("Ejercicio 2 · Parte B")
    r.predicciones({
        "pred_forma_bc": "f8be023c5c429b396b620757effa06ec63bb6a466510856f251b134310496a86",
        "pred_bc_error": "bd1dddfaf233665e87fb493c0364dc67523ff4525fc194616e411b16fa707f7a",
    })
    r.fin()


def check_ejercicio_3():
    r = _Revision("Ejercicio 3 · Parte A")
    t = _tabla()
    cols = _transpuesta(t)
    _arr(r, "total_tienda", [sum(f) for f in t], "debería haber un total por tienda (suma de cada fila)")
    _arr(r, "total_mes", [sum(c) for c in cols], "debería haber un total por mes (suma de cada columna)")
    _esc(r, "total_general", sum(sum(f) for f in t), "suma todos los valores de la tabla")
    idx = [f.index(max(f)) for f in t]
    _arr(r, "mejor_mes_idx", idx, "para cada tienda, la posición del mes con mayor venta")
    _arr(r, "mejor_mes_nombre", [_L["meses"][i] for i in idx], "usa `mejor_mes_idx` para indexar `meses`", tipos="U")
    proms = [statistics.fmean(f) for f in t]
    _arr(r, "promedio_tienda_col", [[p] for p in proms], "debería ser el promedio de cada tienda, en forma de columna `(5, 1)`")
    _arr(r, "desvio_propio", [[x - p for x in f] for f, p in zip(t, proms)],
         "a cada venta réstale el promedio de su propia tienda")
    _arr(r, "participacion", [[x / sum(c) for x, c in zip(f, cols)] for f in t],
         "cada venta entre el total de su mes, para que cada columna sume 1", tol=1e-9)
    r.fin()
    r = _Revision("Ejercicio 3 · Parte B")
    r.predicciones({
        "pred_forma_axis0": "5cf9b00248ab1adbc1c7b3ed197c8c002855618988ec438c6fedc6484401c8c1",
        "pred_forma_keep": "1e3a7a8b25b9a4fe87175855e2297075978da39f86b99b1978779fb8eee1781b",
    })
    r.fin()


def check_ejercicio_4():
    r = _Revision("Ejercicio 4")
    m = _L["clientes_feat"]
    cols = _transpuesta(m)
    _arr(r, "medias", [statistics.fmean(c) for c in cols], "debería haber una media por variable (columna)")
    _arr(r, "desvs", [statistics.pstdev(c) for c in cols], "una desviación estándar poblacional por columna")
    _arr(r, "z", _z_columnas(m), "a cada columna réstale su media y divide entre su desviación", tol=1e-9)
    z = globals().get("z")
    if isinstance(z, np.ndarray) and z.shape == np.array(m).shape:
        if np.allclose(z.mean(axis=0), 0) and np.allclose(z.std(axis=0), 1):
            r.ok("Cada columna de `z` tiene media 0 y desviación 1.")
        else:
            r.mal("Las columnas de `z` deberían quedar con media 0 y desviación 1.")
    f = r.funcion("zscore")
    if f is not _FALTA:
        constante = [[1.0, 7.0], [2.0, 7.0], [3.0, 7.0]]
        casos = [("clientes_feat", np.array(m)), ("<columna constante>", np.array(constante)),
                 ("<una sola fila>", np.array([[4.0, 5.0, 6.0]])), ("<2 x 2>", np.array([[1.0, -1.0], [3.0, 1.0]]))]
        for texto, arr in casos:
            esperado = _z_columnas(arr.tolist())

            def igual(a, b):
                return isinstance(a, np.ndarray) and a.shape == np.array(b).shape and np.allclose(a, b, atol=1e-9)
            r.caso(f"zscore({texto})", f, (arr,), esperado=esperado, igual=igual,
                   motivo="las columnas con desviación 0 deberían quedar en 0 (sin `nan`)" if "constante" in texto or "fila" in texto
                   else "no es lo esperado (debería ser un array de la misma forma)")
    _sin_cambios(r, "clientes_feat")
    r.fin()


def check_reto():
    r = _Revision("Reto final")
    s = _L["saldos"]
    cols = _transpuesta(s)
    proms = [statistics.fmean(f) for f in s]
    _arr(r, "saldo_prom_cliente", [round(p, 2) for p in proms], "promedio de cada cliente (fila), con 2 decimales", tol=0.0051)
    _arr(r, "mes_minimo", [f.index(min(f)) + 1 for f in s], "para cada cliente, el número de mes (1 a 12) con su menor saldo")
    _arr(r, "total_mes_miles", [round(sum(c) / 1000, 1) for c in cols], "total de cada mes (columna) en miles, con 1 decimal", tol=0.051)
    _arr(r, "share_mes", [[x / sum(c) for x, c in zip(f, cols)] for f in s],
         "cada saldo entre el total de su mes, para que cada columna sume 1", tol=1e-9)
    _arr(r, "meses_bajo_promedio", [len([x for x in f if x < p]) for f, p in zip(s, proms)],
         "para cada cliente, cuántos meses estuvo por debajo de su propio promedio")
    z = _z_columnas(s)
    _arr(r, "z_mes", z, "estandariza cada mes (columna) con su media y su desviación", tol=1e-9)
    _esc(r, "n_atipicos", len([x for f in z for x in f if abs(x) > 1.5]), "cuenta los valores de `z_mes` con valor absoluto mayor que 1.5")
    desv = [statistics.pstdev(f) for f in s]
    _esc(r, "cliente_mas_estable", desv.index(min(desv)), "la posición del cliente (fila) con menor desviación de su saldo")
    _sin_cambios(r, "saldos")
    r.fin()


def check_pro():
    r = _Revision("Nivel pro")
    cols = _transpuesta(_L["clientes_feat"])
    mm = [[(x - min(c)) / (max(c) - min(c)) for x in c] for c in cols]
    _arr(r, "minmax", _transpuesta(mm), "cada columna debería ir de 0 (su mínimo) a 1 (su máximo)", tol=1e-9)
    _arr(r, "tabla_precios", [[p * q for q in range(1, 11)] for p in _L["precios_prod"]],
         "una fila por producto y una columna por cantidad, de 1 a 10")
    r.fin()


print("✅ Setup listo. Datos generados y verificadores cargados.")

### 📦 Tus datos de hoy
Los valores se generan con una semilla fija, así que siempre salen iguales.

In [ ]:
print("🏪 Ventas de tiendas")
print("nombres_tiendas  =", nombres_tiendas)
print("meses            =", meses)
print("ventas_flat      =", ventas_flat, ventas_flat.shape)
print("metas_mes        =", metas_mes)
print("descuento_tienda =", descuento_tienda)
print("productos        =", productos, "| precios_prod =", precios_prod)
print("unidades_tp (tiendas × productos):")
print(unidades_tp)
print()
print("🏦 Clientes de un banco")
print("variables =", variables)
print("clientes_feat (8 clientes × 4 variables):")
print(clientes_feat)
print("saldos (6 clientes × 12 meses):")
print(saldos)

---
## 1. Crear arrays 2D, `reshape` y transpuesta

### 📘 Concepto
Un array 2D es una tabla: `shape` es `(filas, columnas)`.
- Se crea con una lista de listas (`np.array([[1, 2], [3, 4]])`) o con una tupla de forma: `np.zeros((3, 4))`.
- `a.reshape(filas, columnas)` reordena los mismos datos en otra forma. Se llena **fila por fila** y el total de elementos tiene que coincidir.
- En `reshape`, un `-1` significa "calcula tú esta dimensión": `a.reshape(-1, 6)` con 30 elementos da `(5, 6)`.
- `a.reshape(-1, 1)` convierte un array 1D en una **columna**, de forma `(n, 1)`.
- `a.T` es la **transpuesta**: las filas pasan a ser columnas.

In [ ]:
numeros_ej = np.arange(1, 13)
m_ej = numeros_ej.reshape(3, 4)
print(m_ej, m_ej.shape)
print(numeros_ej.reshape(-1, 6).shape)     # NumPy calcula que son 2 filas
print(m_ej.T, m_ej.T.shape)
print(np.array([10, 20, 30]).reshape(-1, 1))
# numeros_ej.reshape(5, -1)   # ValueError: 12 elementos no se reparten en 5 filas

### ✍️ Tu turno · Ejercicio 1: armar la tabla de ventas
**Parte A.** `ventas_flat` tiene 30 ventas mensuales: primero los 6 meses de la primera tienda, luego los 6 de la segunda, y así.
1. `tabla`: una matriz de 5 filas (tiendas) por 6 columnas (meses).
2. `tabla_auto`: la misma matriz, indicando solo las 6 columnas y dejando que NumPy calcule las filas.
3. `por_mes`: la transpuesta de `tabla` (meses × tiendas).
4. `columna`: `ventas_flat` convertido en una columna de 30 filas.
5. `ceros_2d`: una matriz de ceros de 3 filas por 4 columnas.

**Parte B.** Predice **sin ejecutar**:

| Variable | Pregunta | Formato |
|---|---|---|
| `pred_reshape_4` | `ventas_flat.reshape(4, -1).shape` | tupla o `"error"` |
| `pred_forma_t` | `np.zeros((3, 4)).T.shape` | tupla, por ejemplo `(2, 5)` |
| `pred_size_3d` | `np.ones((2, 3, 4)).size` | número |
| `pred_ndim_col` | `np.arange(5).reshape(-1, 1).ndim` | número |

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_1()

<details><summary>💡 Pista 1</summary>

`reshape` recibe primero las filas y luego las columnas. La forma de `np.zeros` va entre paréntesis dobles: es una tupla.
</details>

<details><summary>💡 Pista 2</summary>

Para `tabla_auto`, el `-1` va en el lugar de las filas. Para `columna`, el `-1` va en las filas y el `1` en las columnas.
</details>

---
## 2. Broadcasting en 2D: fila y columna contra matriz

### 📘 Concepto
NumPy puede operar arrays de formas distintas si "encajan" **comparando las dimensiones desde la derecha**: cada par tiene que ser igual, o uno de los dos tiene que ser 1.

| Matriz | Otro array | ¿Encaja? | Qué pasa |
|---|---|---|---|
| `(5, 6)` | `(6,)` | sí | la fila se aplica a **cada fila** (un valor por columna) |
| `(5, 6)` | `(5, 1)` | sí | la columna se aplica a **cada columna** (un valor por fila) |
| `(5, 6)` | `(5,)` | **no** | 5 no encaja con 6: `ValueError` |

Por eso, para aplicar un valor distinto a cada **fila**, primero lo conviertes en columna con `reshape(-1, 1)`.

In [ ]:
m_ej = np.array([[10, 20, 30],
                 [40, 50, 60]])            # forma (2, 3)
por_columna_ej = np.array([1, 2, 3])       # forma (3,): un valor por columna
por_fila_ej = np.array([100, 200])         # forma (2,): un valor por fila

print(m_ej + por_columna_ej)
print(m_ej + por_fila_ej.reshape(-1, 1))   # convertido en columna (2, 1)
# m_ej + por_fila_ej   # ValueError: (2, 3) y (2,) no encajan

### ✍️ Tu turno · Ejercicio 2: metas, precios y descuentos
**Parte A.** Usa `tabla` del ejercicio 1.
1. `ingresos_tp`: `unidades_tp` (tiendas × productos) por el precio de cada producto (`precios_prod`).
2. `vs_meta`: cuánto le sobra o le falta a cada venta respecto de la meta de **su mes** (`metas_mes`).
3. `alcanzo_meta`: matriz de `bool`, `True` donde la venta alcanza la meta de su mes.
4. `con_descuento`: cada venta de `tabla` con el descuento de **su tienda** (`descuento_tienda`, uno por fila) aplicado.

**Parte B.** Predice **sin ejecutar**:

| Variable | Pregunta | Formato |
|---|---|---|
| `pred_forma_bc` | `(np.ones((3, 1)) + np.ones((1, 4))).shape` | tupla o `"error"` |
| `pred_bc_error` | `(np.ones((5, 6)) + np.ones(5)).shape` | tupla o `"error"` |

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_2()

<details><summary>💡 Pista 1</summary>

Imprime la forma de cada operando. `precios_prod` y `metas_mes` ya tienen un valor por columna; `descuento_tienda` tiene un valor por fila.
</details>

<details><summary>💡 Pista 2</summary>

Para `con_descuento`: multiplica `tabla` por `(1 - descuento)` con el descuento convertido en columna de forma `(5, 1)`.
</details>

---
## 3. `axis=0`, `axis=1` y `keepdims`

### 📘 Concepto
Las agregaciones (`sum`, `mean`, `max`, `argmax`, `std`...) aceptan `axis`:
- **`axis=0`** recorre las filas hacia abajo y **colapsa las filas**: queda un resultado **por columna**.
- **`axis=1`** recorre las columnas hacia la derecha y **colapsa las columnas**: queda un resultado **por fila**.
- Sin `axis`, resume la tabla entera en un solo número.

Regla para recordarlo: el `axis` que pasas es la dimensión que **desaparece** de la forma. De `(5, 6)`, `axis=0` deja `(6,)` y `axis=1` deja `(5,)`.

`keepdims=True` mantiene la dimensión con tamaño 1: con `axis=1` sale `(5, 1)`, que es justo la forma que encaja para operar de vuelta contra la matriz por filas.

In [ ]:
m_ej = np.array([[10, 20, 30],
                 [40, 50, 60]])
print(m_ej.sum(), m_ej.sum(axis=0), m_ej.sum(axis=1))
print(m_ej.argmax(axis=1))                          # columna del máximo en cada fila
promedio_fila_ej = m_ej.mean(axis=1, keepdims=True)
print(promedio_fila_ej, promedio_fila_ej.shape)
print(m_ej - promedio_fila_ej)                     # cada fila menos su promedio
print(m_ej / m_ej.sum(axis=0))                     # cada columna suma 1

### ✍️ Tu turno · Ejercicio 3: totales, mejores meses y participación
**Parte A.** Con `tabla` (tiendas × meses):
1. `total_tienda`: el total de cada tienda en el semestre.
2. `total_mes`: el total de cada mes, sumando las tiendas.
3. `total_general`: el total de toda la tabla.
4. `mejor_mes_idx`: para cada tienda, la posición del mes con mayor venta; y `mejor_mes_nombre`: los nombres de esos meses, tomados de `meses`.
5. `promedio_tienda_col`: el promedio de cada tienda, con forma `(5, 1)`.
6. `desvio_propio`: cada venta menos el promedio de **su** tienda.
7. `participacion`: qué parte del total del mes aporta cada tienda (cada columna debe sumar 1).

**Parte B.** Predice **sin ejecutar**:

| Variable | Pregunta |
|---|---|
| `pred_forma_axis0` | `np.ones((4, 7)).sum(axis=0).shape` |
| `pred_forma_keep` | `np.ones((4, 7)).sum(axis=1, keepdims=True).shape` |

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_3()

<details><summary>💡 Pista 1</summary>

Pregúntate en cada punto: ¿quiero un resultado por tienda (fila) o por mes (columna)? Por fila es `axis=1`; por columna, `axis=0`.
</details>

<details><summary>💡 Pista 2</summary>

`desvio_propio` es `tabla` menos `promedio_tienda_col`: por eso hacía falta `keepdims`. `participacion` divide `tabla` entre `total_mes`.
</details>

---
## 4. Estandarizar columnas: el z-score

### 📘 Concepto
Las variables de un dataset suelen tener escalas muy distintas (ingresos en miles, años de antigüedad en unidades). El **z-score** las lleva a una escala común: cuántas desviaciones estándar se aleja cada valor del promedio **de su columna**.

```
z = (x - media_de_la_columna) / desviación_de_la_columna
```

Con broadcasting se hace para todas las columnas a la vez: la media y la desviación con `axis=0` tienen forma `(columnas,)` y encajan contra la matriz. Después, cada columna queda con media 0 y desviación 1. Un `|z|` grande (por ejemplo, mayor que 2) señala un valor atípico.

Caso borde: si una columna es constante, su desviación es 0 y la división da `nan`. Una función robusta debería dejar esas columnas en 0.

In [ ]:
m_ej = np.array([[1000.0, 1.0],
                 [3000.0, 2.0],
                 [2000.0, 6.0]])
medias_ej = m_ej.mean(axis=0)
desvs_ej = m_ej.std(axis=0)
z_ej = (m_ej - medias_ej) / desvs_ej
print(np.round(z_ej, 3))
print(np.round(z_ej.mean(axis=0), 10), z_ej.std(axis=0))

### ✍️ Tu turno · Ejercicio 4: clientes en la misma escala
`clientes_feat` tiene 8 clientes (filas) y 4 variables (columnas, ver `variables`).
1. `medias` y `desvs`: la media y la desviación estándar de cada variable.
2. `z`: la matriz estandarizada por columna.
3. `zscore(matriz)`: una función que estandarice por columna cualquier matriz 2D y devuelva un array de la misma forma. Si una columna tiene desviación 0, sus valores deben quedar en 0 (no `nan`).

El verificador probará `zscore` con una columna constante y con una matriz de una sola fila.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_4()

<details><summary>💡 Pista 1</summary>

Los tres primeros puntos son tres líneas con `axis=0`. Para la función, copia esa lógica usando el parámetro `matriz`.
</details>

<details><summary>💡 Pista 2</summary>

Para evitar la división entre 0, antes de dividir reemplaza las desviaciones que valen 0 por 1 con `np.where(desv == 0, 1, desv)`. Así esa columna queda en `(x - media) / 1`, que es 0 porque todos sus valores son iguales a la media.
</details>

---
## 🏋️ Reto final: saldos de clientes en el año
`saldos` tiene el saldo de fin de mes de 6 clientes (filas) durante 12 meses (columnas). Resuelve sin bucles:
1. `saldo_prom_cliente`: el saldo promedio de cada cliente, con 2 decimales.
2. `mes_minimo`: para cada cliente, el **número de mes** (del 1 al 12) en que tuvo su menor saldo.
3. `total_mes_miles`: el total de saldos de cada mes, en miles, con 1 decimal.
4. `share_mes`: qué parte del total de cada mes corresponde a cada cliente (cada columna suma 1).
5. `meses_bajo_promedio`: para cada cliente, en cuántos meses su saldo estuvo por debajo de **su propio** promedio.
6. `z_mes`: los saldos estandarizados **por mes** (cada columna con su media y su desviación), para comparar a los clientes dentro de cada mes.
7. `n_atipicos`: cuántos valores de `z_mes` tienen valor absoluto mayor que 1.5.
8. `cliente_mas_estable`: la posición del cliente cuyo saldo varió menos durante el año (menor desviación estándar).

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_reto()

<details><summary>💡 Pista 1</summary>

Antes de cada punto decide si el resultado es por cliente (fila, `axis=1`) o por mes (columna, `axis=0`).
</details>

<details><summary>💡 Pista 2</summary>

Para `meses_bajo_promedio`: compara `saldos` con el promedio por fila con `keepdims=True` y suma la máscara con `axis=1`. Para `n_atipicos`, usa `np.abs` y cuenta con `np.sum`.
</details>

---
## 🚀 Nivel pro (opcional)
1. `minmax`: lleva cada columna de `clientes_feat` a la escala 0–1 con `(x - mínimo) / (máximo - mínimo)`, usando mínimos y máximos por columna.
2. `tabla_precios`: una tabla de 4 filas (productos) y 10 columnas con el precio de comprar de 1 a 10 unidades de cada producto, usando `precios_prod` como columna y `np.arange(1, 11)` como fila. Sin bucles.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_pro()

---
## ✅ Cierre: autoevaluación
Marca lo que puedes hacer sin mirar:
- [ ] Crear un array 2D y explicar qué dice su `shape`.
- [ ] Usar `reshape` con `-1` y convertir un array 1D en columna.
- [ ] Explicar qué hace la transpuesta.
- [ ] Decidir si dos formas encajan para el broadcasting, comparando desde la derecha.
- [ ] Aplicar un valor distinto a cada fila y a cada columna de una matriz.
- [ ] Explicar qué dimensión desaparece con `axis=0` y con `axis=1`.
- [ ] Usar `keepdims=True` para volver a operar contra la matriz.
- [ ] Estandarizar las columnas de una matriz y proteger las columnas constantes.

**Próxima sesión (S07):** indexing 2D, datos sucios, cargar archivos y el primer avance del proyecto.